In [1]:
import os
import mlflow
import matplotlib.pyplot as plt

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc
)

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

In [3]:
# =========================================================
# Columns
# =========================================================

nomod_columns = [
    'HasCrCard',
    'IsActiveMember',
    'Complain'
]

dummyfy_columns = [
    'Card Type',
    'NumOfProducts',
    'Geography',
    'Gender'
]

norm_std_columns = [
    'Balance',
    'Point Earned',
    'CreditScore',
    'Age',
    'Tenure',
    'Satisfaction Score',
    'EstimatedSalary'
]

In [4]:
# =========================================================
# Config
# =========================================================

RANDOM_STATE = 42
EXPERIMENT_NAME = "customer-churn-simple"

In [5]:
# =========================================================
# Paths
# =========================================================

ROOT_DIR = os.path.abspath("../")

DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")

ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# =========================================================
# MLflow
# =========================================================

mlflow.set_tracking_uri(
    f"sqlite:///{DB_PATH}"
)

# Create experiment only once
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:

    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_DIR}"
    )

mlflow.set_experiment(EXPERIMENT_NAME)

2026/05/19 16:53:26 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/19 16:53:26 INFO mlflow.store.db.utils: Updating database tables


<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1779231206936, experiment_id='1', last_update_time=1779231206936, lifecycle_stage='active', name='customer-churn-simple', tags={}, trace_location=None, workspace='default'>

In [6]:
# =========================================================
# Preprocessor
# =========================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                drop='first'
            ),
            dummyfy_columns
        ),
        (
            'num',
            StandardScaler(),
            norm_std_columns
        ),
        (
            'pass',
            'passthrough',
            nomod_columns
        )
    ],
    remainder='drop'
)


In [7]:
# =========================================================
# Models
# =========================================================

models = {
    "logistic_regression": LogisticRegression(
        random_state=RANDOM_STATE,
        max_iter=2000
    ),

    "decision_tree": DecisionTreeClassifier(
        random_state=RANDOM_STATE
    ),

    "random_forest": RandomForestClassifier(
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [8]:
# =========================================================
# Helper: get names
# =========================================================

def get_feature_names(preprocessor):
    return preprocessor.get_feature_names_out()

# =========================================================
# Helper: Log Curves
# =========================================================

def log_classification_curves(y_test, y_proba):

    # --------------------------------
    # ROC Curve
    # --------------------------------

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = roc_auc_score(y_test, y_proba)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"ROC AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend(loc="lower right")

    mlflow.log_figure(plt.gcf(), "roc_curve.png")
    plt.close()

    # --------------------------------
    # Precision Recall Curve
    # --------------------------------

    precision, recall, _ = precision_recall_curve(
        y_test,
        y_proba
    )

    pr_auc = average_precision_score(y_test, y_proba)

    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, label=f"PR AUC = {pr_auc:.4f}")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend(loc="lower left")

    mlflow.log_figure(plt.gcf(), "pr_curve.png")
    plt.close()

In [9]:
# =========================================================
# Train Log
# =========================================================
def train_log(models, preprocessor, X_train, y_train, X_test, y_test):
    for model_name, model in models.items():

        with mlflow.start_run(run_name=model_name):

            # --------------------------------
            # Build Pipeline
            # --------------------------------

            pipeline = Pipeline([
                ('preprocessing', preprocessor),
                ('model', model)
            ])

            # --------------------------------
            # Train
            # --------------------------------

            pipeline.fit(X_train, y_train)

            # --------------------------------
            # Predictions
            # --------------------------------

            y_pred = pipeline.predict(X_test)

            if hasattr(pipeline, "predict_proba"):

                y_proba = pipeline.predict_proba(X_test)[:, 1]

                roc_auc = roc_auc_score(y_test, y_proba)

                pr_auc = average_precision_score(
                    y_test,
                    y_proba
                )

                # Log plots
                log_classification_curves(
                    y_test,
                    y_proba
                )

            else:
                roc_auc = None
                pr_auc = None

            # --------------------------------
            # Metrics
            # --------------------------------

            metrics = {
                "accuracy": accuracy_score(y_test, y_pred),
                "precision": precision_score(y_test, y_pred),
                "recall": recall_score(y_test, y_pred),
                "f1_score": f1_score(y_test, y_pred)
            }

            if roc_auc is not None:
                metrics["roc_auc"] = roc_auc

            if pr_auc is not None:
                metrics["pr_auc"] = pr_auc

            mlflow.log_metrics(metrics)

            # --------------------------------
            # Params
            # --------------------------------

            mlflow.log_params(model.get_params())

            # --------------------------------
            # Feature Schema
            # --------------------------------

            fitted_preprocessor = pipeline.named_steps['preprocessing']

            input_features = (
                dummyfy_columns +
                norm_std_columns +
                nomod_columns
            )

            output_features = get_feature_names(
                fitted_preprocessor
            )

            mlflow.log_dict(
                {
                    "input_features": input_features,
                    "output_features": output_features.tolist()
                },
                "feature_schema.json"
            )

            # --------------------------------
            # Reports
            # --------------------------------

            mlflow.log_dict(
                classification_report(
                    y_test,
                    y_pred,
                    output_dict=True
                ),
                "classification_report.json"
            )

            mlflow.log_dict(
                confusion_matrix(
                    y_test,
                    y_pred
                ).tolist(),
                "confusion_matrix.json"
            )

            # --------------------------------
            # Log Model
            # --------------------------------

            mlflow.sklearn.log_model(
                sk_model=pipeline,
                name="model",
                serialization_format="skops"
            )

            print(f"Finished: {model_name}")

In [10]:
train_log(models, preprocessor, X_train, y_train, X_test, y_test)

Finished: logistic_regression
Finished: decision_tree
Finished: random_forest


In [11]:
def experiment_metrics(experiment_name):
    # --------------------------------
    # Get Experiment
    # --------------------------------

    experiment = mlflow.get_experiment_by_name(experiment_name)
    experiment_id = experiment.experiment_id

    # --------------------------------
    # Search Runs
    # --------------------------------

    runs_df = mlflow.search_runs(
        experiment_ids=[experiment_id]
    )

    # Display summary table
    summary_df = runs_df[[
            "run_id",
            "tags.mlflow.runName",
            "metrics.accuracy",
            "metrics.precision",
            "metrics.recall",
            "metrics.f1_score",
            "metrics.roc_auc",
            "metrics.pr_auc",
            "start_time"
        ]]

    # --------------------------------
    # Sort by Best Model
    # --------------------------------

    summary_df = summary_df.sort_values(
        by="metrics.pr_auc",
        ascending=False
    )


    # --------------------------------
    # Display
    # --------------------------------

    display(summary_df)

In [12]:
experiment_metrics(EXPERIMENT_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc,start_time
2,c7cd06bd171e48499fbfc19b03b4ab31,logistic_regression,0.9980,0.990291,1.0,0.995122,0.999786,0.999112,2026-05-19 22:53:27.045000+00:00
0,0296447b1a9f4dfda3cd4f85a8247504,random_forest,0.9980,0.990291,1.0,0.995122,0.999770,0.998888,2026-05-19 22:53:36.618000+00:00
1,dfeeaa834884442db7041098d6beeefe,decision_tree,0.9975,0.987893,1.0,0.993910,0.998430,0.987893,2026-05-19 22:53:33.197000+00:00


# Modeling Approach and Key Design Choices

This churn modeling experiment evaluates three baseline classifiers—Logistic Regression, Decision Tree, and Random Forest—within a consistent preprocessing and evaluation framework. The goal is to identify a robust model that generalizes well while keeping interpretability and deployment simplicity in mind.

1. ## Preprocessing Strategy

A ColumnTransformer was used to ensure consistent and leakage-free preprocessing across all models:

One-Hot Encoding (dummyfy_columns)
Applied to: Card Type, NumOfProducts, Geography, Gender
drop='first' was used to avoid multicollinearity and reduce redundancy.
Standard Scaling (norm_std_columns)
Applied to continuous numerical features such as Balance, Age, CreditScore, etc.
This is particularly important for linear models (Logistic Regression) to ensure stable optimization and comparable coefficient scales.
Passthrough features (nomod_columns)
Binary or already encoded features such as HasCrCard, IsActiveMember, and Complain were left unchanged.

This setup ensures each feature type is treated appropriately while maintaining a single unified pipeline.

2. ## Model Selection Rationale

Three models were chosen to represent increasing complexity:

Logistic Regression
Serves as a strong, interpretable baseline.
Useful for understanding directional impact of features.
Regularization and scaling make it stable and efficient.
Decision Tree
Captures non-linear relationships and feature interactions.
Highly interpretable but prone to overfitting.
Random Forest
Ensemble method that reduces overfitting compared to a single tree.
Typically strong performance on tabular data without heavy tuning.

All models were trained with a fixed random_state=42 for reproducibility.

3. ## Evaluation Strategy and Key Metric Choice

Given the nature of churn prediction (often involving class imbalance and business-sensitive false positives/negatives), multiple metrics were tracked:

Primary Metric: PR-AUC (Average Precision Score)
Selected as the most important metric.
More informative than accuracy in imbalanced settings.
Focuses on performance for the positive class (churners), which is usually the business target.
Secondary Metrics
ROC-AUC: Measures ranking quality across thresholds.
Precision: Important to reduce false churn predictions (avoid unnecessary interventions).
Recall: Ensures actual churners are identified.
F1-score: Balance between precision and recall.
Accuracy: Included but not relied upon due to potential imbalance distortion.

Additionally, ROC and Precision-Recall curves were logged to MLflow for visual inspection of model behavior across thresholds.

4. ## Experiment Tracking and Reproducibility

All experiments were tracked using MLflow:

Parameters logged: full model hyperparameters (get_params()).
Metrics logged: accuracy, precision, recall, F1, ROC-AUC, PR-AUC.
Artifacts logged:
Classification report
Confusion matrix
ROC and PR curves
Feature schema (input/output after preprocessing)
Serialized model pipeline

This ensures full reproducibility and traceability of results.

5. ## Results Summary and Model Selection Insight

All three models achieved extremely high performance:

Logistic Regression achieved the best PR-AUC (~0.999)
Random Forest performed nearly identically
Decision Tree slightly underperformed in ranking-based metrics
Key Observation

The very high scores across all models suggest:

The dataset likely contains highly predictive or potentially leaky features
Or the problem is relatively easy / separable in feature space

Because Logistic Regression achieved the highest PR-AUC with similar performance to more complex models, it becomes the preferred candidate due to:

Simplicity
Interpretability
Competitive performance
Lower risk of overfitting compared to ensembles

6. ## Final Model Choice

Logistic Regression was selected as the final model because it:

Achieves the best PR-AUC (primary metric)
Matches or exceeds more complex models in all key metrics
Provides interpretability for business stakeholders
Integrates cleanly into production pipelines

## Note

Given the near-perfect performance across models, there is a strong indication of potential target leakage. In particular, the feature Complain may be directly or indirectly correlated with the churn outcome in a way that would not be available at prediction time in a real production setting.

To address this, Complain will be excluded from the feature set, and all models will be retrained and reevaluated. This will ensure a more realistic assessment of model performance and improve the validity of the experimental results.

Additionally, the updated experiment will help confirm whether the remaining features provide sufficient predictive power without leakage-driven signals.

In [13]:
# =========================================================
# Config
# =========================================================

RANDOM_STATE = 42
EXPERIMENT_NAME = "customer-churn-simple-no-complain-feature"

In [14]:
# =========================================================
# MLflow
# =========================================================

mlflow.set_tracking_uri(
    f"sqlite:///{DB_PATH}"
)

# Create experiment only once
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:

    mlflow.create_experiment(
        name=EXPERIMENT_NAME,
        artifact_location=f"file://{ARTIFACTS_DIR}"
    )

mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1779231220795, experiment_id='2', last_update_time=1779231220795, lifecycle_stage='active', name='customer-churn-simple-no-complain-feature', tags={}, trace_location=None, workspace='default'>

In [15]:
# =========================================================
# Columns
# =========================================================

nomod_columns = [
    'HasCrCard',
    'IsActiveMember'
]
# =========================================================
# Preprocessor
# =========================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                drop='first'
            ),
            dummyfy_columns
        ),
        (
            'num',
            StandardScaler(),
            norm_std_columns
        ),
        (
            'pass',
            'passthrough',
            nomod_columns
        )
    ],
    remainder='drop'
)

In [16]:
train_log(models, preprocessor, X_train, y_train, X_test, y_test)

Finished: logistic_regression
Finished: decision_tree
Finished: random_forest


In [17]:
experiment_metrics(EXPERIMENT_NAME)

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc,metrics.pr_auc,start_time
0,42d00bf3ec564f219d79ea567c2abde4,random_forest,0.8755,0.795539,0.524510,0.632201,0.875317,0.733387,2026-05-19 22:53:48.559000+00:00
2,02b192f39f6b4fe8bfcb2f24e0cef945,logistic_regression,0.8530,0.743590,0.426471,0.542056,0.838354,0.649413,2026-05-19 22:53:40.806000+00:00
1,4a2364a883534935b83d316f887416f8,decision_tree,0.7965,0.501080,0.568627,0.532721,0.711763,0.372928,2026-05-19 22:53:44.593000+00:00


- Random Forest (Best Overall Model):
    - PR-AUC: 0.733387 (best)
    - ROC-AUC: 0.875317
    - F1-score: 0.632201
    - Best balance between ranking performance and classification quality
    - Handles non-linear interactions better than linear models

- Logistic Regression:
    - PR-AUC: 0.649413
    - ROC-AUC: 0.838354	
    - More stable but underfits non-linear patterns
    - Lower recall suggests limited ability to capture churn complexity

- Decision Tree:
    - PR-AUC: 0.372928 (significantly worse)
    - High recall but very low precision stability
    - Indicates overfitting and poor generalization

### The gap between models is now meaningful, unlike before:

 - Random Forest clearly benefits from capturing non-linear interactions
 - Logistic Regression provides a strong but simplified baseline
 - Decision Tree alone is too unstable without ensembling

This confirms that once leakage is removed, the problem becomes a realistic tabular churn prediction task rather than an artificially separable classification problem.

Removing Complain significantly improved the validity of the experiment and revealed the true difficulty of the problem.

### Random Forest is now the preferred model, as it achieves the best trade-off between:

 - PR-AUC (primary metric)
 - Recall of churners
 - Robustness to feature interactions

MLflow Experiment
    └── Optuna Study
            └── Trial Loop
                    ├── Choose hyperparameters (Bayesian optimization)
                    ├── Build preprocessing + model pipeline
                    ├── Run Cross Validation
                    ├── Compute ROC-AUC
                    ├── Report metric to Optuna
                    ├── Optuna decides:
                    │       ├── continue
                    │       └── prune trial
                    └── MLflow logs params + metrics
            └── Best trial selected
    └── Train best model on full training data
    └── Save model to MLflow